## Example 1: Basic WebSocket Room and Document Setup

In [ ]:
from pycrdt.websocket import WebsocketServer, YRoom
import pycrdt

# Example 1: Basic WebSocket Room and Document Setup
print("Example 1 - Basic WebSocket Room and Document Setup")

# Create a collaborative document room
room = YRoom()
print(f"Created YRoom: {type(room).__name__}")

# Create a document for collaboration
doc = pycrdt.Doc()
shared_text = doc.get("text", type=pycrdt.Text)
shared_text += "Hello collaborative world!"

print(f"Document text: '{str(shared_text)}'")

# Get document update for synchronization
update = doc.get_update()
print(f"Document update size: {len(update)} bytes")

# Show room capabilities
room_methods = [m for m in dir(room) if not m.startswith('_')]
print(f"\nYRoom has {len(room_methods)} public methods")
print(f"First 5 methods: {room_methods[:5]}")

print("\nExample 1 completed successfully")

## Example 2: Multi-client document synchronization simulation

In [ ]:
from pycrdt.websocket import WebsocketServer, YRoom, ASGIServer
import pycrdt
import asyncio

# Example 2: Multi-client document synchronization simulation
print("pycrdt.websocket module imported successfully")

try:
    print("Example 2 - Multi-client document synchronization simulation")
    
    # Simulate multiple clients working on the same document
    class MockClient:
        def __init__(self, name, room):
            self.name = name
            self.room = room
            self.doc = pycrdt.Doc()
            self.messages_sent = []
            self.messages_received = []
            
        def send_update(self, update_data):
            """Simulate sending an update to the room"""
            self.messages_sent.append(f"Update: {len(update_data)} bytes")
            return f"Sent by {self.name}"
            
        def receive_update(self, update_data):
            """Simulate receiving an update from the room"""
            self.doc.apply_update(update_data)
            self.messages_received.append(f"Applied: {len(update_data)} bytes")
            return f"Received by {self.name}"
            
        def get_stats(self):
            return {
                "name": self.name,
                "messages_sent": len(self.messages_sent),
                "messages_received": len(self.messages_received),
                "doc_client_id": self.doc.client_id
            }
    
    # Create a shared room
    room = YRoom()
    print(f"Created shared YRoom for collaboration")
    
    # Create multiple mock clients
    client_alice = MockClient("Alice", room)
    client_bob = MockClient("Bob", room)
    client_charlie = MockClient("Charlie", room)
    
    clients = [client_alice, client_bob, client_charlie]
    print(f"Created {len(clients)} mock clients")
    
    # Each client creates their own document view
    for client in clients:
        shared_array = client.doc.get("tasks", type=pycrdt.Array)
        shared_map = client.doc.get("metadata", type=pycrdt.Map)
        print(f"  {client.name}: Document initialized")
    
    # Alice makes changes
    alice_tasks = client_alice.doc.get("tasks", type=pycrdt.Array)
    alice_tasks.extend(["Task 1: Setup project", "Task 2: Design UI"])
    alice_update = client_alice.doc.get_update()
    client_alice.send_update(alice_update)
    
    print(f"\nAlice added tasks: {list(alice_tasks)}")
    
    # Simulate Alice's update being broadcast to other clients
    for client in [client_bob, client_charlie]:
        client.receive_update(alice_update)
        client_tasks = client.doc.get("tasks", type=pycrdt.Array)
        print(f"  {client.name} received tasks: {list(client_tasks)}")
    
    # Bob adds more content
    bob_tasks = client_bob.doc.get("tasks", type=pycrdt.Array)
    bob_metadata = client_bob.doc.get("metadata", type=pycrdt.Map)
    
    # Get state before Bob's changes
    bob_state_before = client_alice.doc.get_state()
    
    bob_tasks.append("Task 3: Implement backend")
    bob_metadata["created_by"] = "Bob"
    bob_metadata["timestamp"] = "2024-01-01T10:00:00"
    
    # Get incremental update
    bob_update = client_bob.doc.get_update(bob_state_before)
    client_bob.send_update(bob_update)
    
    print(f"\nBob added content:")
    print(f"  Tasks: {list(bob_tasks)}")
    print(f"  Metadata: {dict(bob_metadata)}")
    
    # Apply Bob's changes to other clients
    for client in [client_alice, client_charlie]:
        client.receive_update(bob_update)
    
    # Charlie makes concurrent changes
    charlie_state_before = client_alice.doc.get_state()
    charlie_tasks = client_charlie.doc.get("tasks", type=pycrdt.Array)
    charlie_metadata = client_charlie.doc.get("metadata", type=pycrdt.Map)
    
    charlie_tasks.append("Task 4: Testing")
    charlie_metadata["reviewed_by"] = "Charlie"
    
    charlie_update = client_charlie.doc.get_update(charlie_state_before)
    client_charlie.send_update(charlie_update)
    
    # Final synchronization - everyone gets everyone's changes
    all_updates = [alice_update, bob_update, charlie_update]
    
    print(f"\nFinal synchronization across all clients:")
    for client in clients:
        for update in all_updates:
            client.receive_update(update)
        
        final_tasks = client.doc.get("tasks", type=pycrdt.Array)
        final_metadata = client.doc.get("metadata", type=pycrdt.Map)
        
        print(f"  {client.name}'s final view:")
        print(f"    Tasks ({len(final_tasks)}): {list(final_tasks)}")
        print(f"    Metadata: {dict(final_metadata)}")
        print(f"    Stats: {client.get_stats()}")
    
    # Verify all clients have identical state
    final_states = []
    for client in clients:
        tasks = list(client.doc.get("tasks", type=pycrdt.Array))
        metadata = dict(client.doc.get("metadata", type=pycrdt.Map))
        final_states.append((tasks, metadata))
    
    all_identical = all(state == final_states[0] for state in final_states)
    print(f"\nAll clients synchronized: {all_identical}")
    
    print("Multi-client document synchronization completed")
    print("pycrdt_websocket example 2 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"pycrdt_websocket example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: User awareness and presence tracking

In [ ]:
from pycrdt.websocket import YRoom, ASGIServer
from pycrdt.websocket.awareness import Awareness
import pycrdt
import asyncio

# Example 3: User awareness and presence tracking
print("pycrdt.websocket module imported successfully")

try:
    print("Example 3 - User awareness and presence tracking in collaborative sessions")
    
    # Create awareness system for tracking user presence
    print("Setting up awareness system for user presence...")
    
    # Document with awareness
    doc = pycrdt.Doc()
    awareness = Awareness(doc)
    
    print(f"Created Awareness system: {type(awareness).__name__}")
    print(f"Awareness client ID: {awareness.client_id}")
    
    # Simulate multiple users with presence information
    users_presence = {
        "alice": {
            "name": "Alice Smith",
            "cursor": {"x": 100, "y": 200},
            "selection": {"start": 0, "end": 5},
            "color": "#ff6b6b",
            "status": "online"
        },
        "bob": {
            "name": "Bob Johnson", 
            "cursor": {"x": 250, "y": 180},
            "selection": {"start": 12, "end": 20},
            "color": "#4ecdc4",
            "status": "typing"
        },
        "charlie": {
            "name": "Charlie Brown",
            "cursor": {"x": 50, "y": 300},
            "selection": None,
            "color": "#45b7d1",
            "status": "idle"
        }
    }
    
    print(f"\nSimulating {len(users_presence)} users with presence data:")
    for user_id, presence in users_presence.items():
        print(f"  {user_id}: {presence['name']} - {presence['status']}")
        print(f"    Cursor: ({presence['cursor']['x']}, {presence['cursor']['y']})")
        if presence['selection']:
            print(f"    Selection: {presence['selection']['start']}-{presence['selection']['end']}")
        print(f"    Color: {presence['color']}")
    
    # Awareness state management
    print(f"\nAwareness state management:")
    
    # Set local state (simulating current user)
    local_state = {
        "name": "Local User",
        "cursor": {"x": 300, "y": 150},
        "color": "#e17055"
    }
    
    # Simulate awareness updates
    awareness_updates = []
    for user_id, presence in users_presence.items():
        # Create mock awareness update
        update_data = {
            "user": user_id,
            "state": presence,
            "timestamp": f"2024-01-01T10:{10 + len(awareness_updates):02d}:00"
        }
        awareness_updates.append(update_data)
    
    print(f"Generated {len(awareness_updates)} awareness updates")
    
    # Process awareness information
    print(f"\nProcessing user awareness information:")
    
    active_users = []
    cursor_positions = []
    
    for update in awareness_updates:
        user_data = update["state"]
        if user_data["status"] in ["online", "typing"]:
            active_users.append(update["user"])
            cursor_positions.append(user_data["cursor"])
    
    print(f"  Active users: {active_users}")
    print(f"  Cursor positions: {cursor_positions}")
    
    # Awareness synchronization simulation
    class AwarenessManager:
        def __init__(self):
            self.states = {}
            self.subscribers = []
            
        def update_user_state(self, client_id, state):
            """Update user's awareness state"""
            self.states[client_id] = state
            self.broadcast_awareness()
            
        def remove_user(self, client_id):
            """Remove user from awareness"""
            if client_id in self.states:
                del self.states[client_id]
                self.broadcast_awareness()
                
        def broadcast_awareness(self):
            """Simulate broadcasting awareness to all clients"""
            return f"Broadcasting to {len(self.subscribers)} clients"
            
        def get_active_users(self):
            """Get list of active users"""
            return list(self.states.keys())
            
        def get_user_cursors(self):
            """Get cursor positions of all users"""
            cursors = {}
            for client_id, state in self.states.items():
                if "cursor" in state:
                    cursors[client_id] = state["cursor"]
            return cursors
    
    # Test awareness manager
    awareness_mgr = AwarenessManager()
    
    # Add users to awareness
    for user_id, presence in users_presence.items():
        awareness_mgr.update_user_state(user_id, presence)
    
    print(f"\nAwareness Manager status:")
    print(f"  Active users: {awareness_mgr.get_active_users()}")
    print(f"  User cursors: {awareness_mgr.get_user_cursors()}")
    
    # Simulate user leaving
    awareness_mgr.remove_user("charlie")
    print(f"  After Charlie left: {awareness_mgr.get_active_users()}")
    
    # Real-time collaboration features enabled by awareness
    collaboration_features = {
        "Live cursors": "Show other users' cursor positions in real-time",
        "Selection highlighting": "Highlight text selected by other users", 
        "User presence indicators": "Show online/offline status of collaborators",
        "Typing indicators": "Show when someone is actively typing",
        "User avatars": "Display user profile pictures/initials",
        "Activity feed": "Track who made what changes and when"
    }
    
    print(f"\nCollaboration features enabled by awareness:")
    for feature, description in collaboration_features.items():
        print(f"  {feature}: {description}")
    
    # Message types for awareness protocol
    message_types = {
        "awareness_update": "User state changes (cursor, selection, etc.)",
        "user_join": "New user joined the document",
        "user_leave": "User disconnected from the document", 
        "heartbeat": "Keep-alive signal from active users"
    }
    
    print(f"\nAwareness protocol message types:")
    for msg_type, description in message_types.items():
        print(f"  {msg_type}: {description}")
    
    print("User awareness and presence tracking completed")
    print("pycrdt_websocket example 3 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"pycrdt_websocket example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: Server integration and deployment patterns

In [ ]:
from pycrdt.websocket import ASGIServer, YRoom, WebsocketServer
import pycrdt
import asyncio

# Example 4: Server integration and deployment patterns
print("pycrdt.websocket module imported successfully")

try:
    print("Example 4 - Server integration and deployment patterns")
    
    # ASGI Server configuration for web framework integration
    print("Configuring ASGI server for web framework integration...")
    
    # Server configuration options
    server_configs = {
        "ASGIServer": {
            "description": "ASGI-compatible server for FastAPI, Django, etc.",
            "use_cases": ["Web applications", "REST APIs", "Full-stack apps"],
            "frameworks": ["FastAPI", "Django", "Starlette", "Quart"],
            "features": ["HTTP/WebSocket support", "Middleware integration", "Authentication"]
        },
        "WebsocketServer": {
            "description": "Standalone WebSocket server for CRDT sync",
            "use_cases": ["Real-time collaboration", "Gaming", "Chat applications"],
            "protocols": ["WebSocket", "Socket.IO compatible", "Custom protocols"],
            "features": ["Low latency", "Direct connection", "Custom routing"]
        }
    }
    
    for server_type, config in server_configs.items():
        print(f"\n{server_type}:")
        print(f"  Description: {config['description']}")
        print(f"  Use cases: {', '.join(config['use_cases'])}")
        if 'frameworks' in config:
            print(f"  Compatible frameworks: {', '.join(config['frameworks'])}")
        if 'protocols' in config:
            print(f"  Supported protocols: {', '.join(config['protocols'])}")
        print(f"  Key features: {', '.join(config['features'])}")
    
    # Room management strategies
    print(f"\nRoom management strategies:")
    
    room_strategies = {
        "Single Document Room": {
            "pattern": "One room per document",
            "scalability": "Good for small teams",
            "use_case": "Document editing, note-taking"
        },
        "Multi-Room Hub": {
            "pattern": "Multiple rooms per application instance",
            "scalability": "Medium scale applications",
            "use_case": "Project management, team workspaces"
        },
        "Distributed Rooms": {
            "pattern": "Rooms across multiple server instances",
            "scalability": "High scale, enterprise",
            "use_case": "Large organizations, global collaboration"
        }
    }
    
    for strategy, details in room_strategies.items():
        print(f"  {strategy}:")
        print(f"    Pattern: {details['pattern']}")
        print(f"    Scalability: {details['scalability']}")
        print(f"    Use case: {details['use_case']}")
    
    # Authentication and security patterns
    print(f"\nAuthentication and security patterns:")
    
    security_patterns = {
        "JWT Token Authentication": {
            "implementation": "Validate JWT tokens on WebSocket connection",
            "security_level": "High",
            "complexity": "Medium"
        },
        "Session-based Auth": {
            "implementation": "Use HTTP session cookies for WebSocket auth",
            "security_level": "Medium",
            "complexity": "Low"
        },
        "API Key Authentication": {
            "implementation": "Authenticate using API keys in headers",
            "security_level": "Medium",
            "complexity": "Low"
        },
        "Role-based Access Control": {
            "implementation": "Check user permissions for document access",
            "security_level": "High",
            "complexity": "High"
        }
    }
    
    for auth_type, details in security_patterns.items():
        print(f"  {auth_type}:")
        print(f"    Implementation: {details['implementation']}")
        print(f"    Security Level: {details['security_level']}")
        print(f"    Complexity: {details['complexity']}")
    
    # Production deployment considerations
    print(f"\nProduction deployment considerations:")
    
    deployment_checklist = [
        "Load balancing across multiple server instances",
        "WebSocket connection scaling and sticky sessions",
        "Document persistence and backup strategies", 
        "Monitoring and logging for collaboration events",
        "Error handling and graceful connection recovery",
        "Rate limiting and abuse prevention",
        "CORS configuration for cross-origin requests",
        "SSL/TLS encryption for secure connections"
    ]
    
    for i, item in enumerate(deployment_checklist, 1):
        print(f"  {i}. {item}")
    
    # Performance optimization strategies
    print(f"\nPerformance optimization strategies:")
    
    optimizations = {
        "Message Batching": "Combine multiple small updates into batches",
        "Delta Compression": "Send only incremental changes, not full state",
        "Client Throttling": "Limit update frequency from high-activity clients",
        "Room Partitioning": "Split large documents into smaller collaborative units",
        "Caching Layer": "Cache frequently accessed document states",
        "Connection Pooling": "Reuse connections efficiently"
    }
    
    for optimization, description in optimizations.items():
        print(f"  {optimization}: {description}")
    
    # Integration example patterns
    print(f"\nCommon integration patterns:")
    
    integration_examples = {
        "FastAPI + WebSocket": {
            "code_pattern": "@app.websocket('/ws/{room_id}')",
            "description": "REST API with WebSocket endpoints for real-time features"
        },
        "Django Channels": {
            "code_pattern": "WebsocketConsumer with YRoom integration",
            "description": "Django applications with real-time collaboration"
        },
        "Standalone Service": {
            "code_pattern": "Dedicated collaboration microservice",
            "description": "Separate service handling all collaborative features"
        }
    }
    
    for pattern, details in integration_examples.items():
        print(f"  {pattern}:")
        print(f"    Pattern: {details['code_pattern']}")
        print(f"    Description: {details['description']}")
    
    print("Server integration and deployment patterns completed")
    print("pycrdt_websocket example 4 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"pycrdt_websocket example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")

## Example 5: Collaborative Application with Multiple Rooms

In [ ]:
from pycrdt.websocket import YRoom
from pycrdt.websocket.awareness import Awareness
import pycrdt

# Example 5: Collaborative Application with Multiple Rooms
print("Example 5 - Collaborative Application with Multiple Rooms")

# Create collaborative application
class CollaborativeApp:
    def __init__(self):
        self.rooms = {}
        
    def create_room(self, room_id):
        room = YRoom()
        doc = pycrdt.Doc()
        self.rooms[room_id] = {"room": room, "doc": doc, "users": []}
        return room_id
        
    def add_content(self, room_id, content):
        if room_id in self.rooms:
            doc = self.rooms[room_id]["doc"]
            text = doc.get("content", type=pycrdt.Text)
            text += content
            return doc.get_update()
        return None

# Initialize app and create rooms
app = CollaborativeApp()
app.create_room("doc1")
app.create_room("doc2")

print(f"Created {len(app.rooms)} rooms")

# Add content to rooms
update1 = app.add_content("doc1", "Document 1 content")
update2 = app.add_content("doc2", "Document 2 content")

print(f"Room 1 update: {len(update1)} bytes")
print(f"Room 2 update: {len(update2)} bytes")

# Show room statistics
for room_id, data in app.rooms.items():
    doc = data["doc"]
    text = doc.get("content", type=pycrdt.Text)
    print(f"{room_id}: {len(str(text))} characters")

print("\nExample 5 completed successfully")